# Stage 1 · RL Foundations — EXERCISES
### Topics: MDPs · Bellman Equations · Policy Gradient Theorem · Advantage & GAE · REINFORCE · A2C · PPO

> Fill every `# TODO`. Run the `# ASSERT` cells to verify.  
> Implement each algorithm from scratch — these are the primitives everything else builds on.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from typing import List, Tuple, Optional
from dataclasses import dataclass, field


ModuleNotFoundError: No module named 'torch'

---
## 1 · Markov Decision Process & Discounted Returns

An MDP is a tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:
- $\mathcal{S}$ — state space; $\mathcal{A}$ — action space
- $P(s'|s,a)$ — transition dynamics
- $R(s,a)$ — reward function
- $\gamma \in [0,1)$ — discount factor (controls myopia)

### Discounted return from timestep t
$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots = \sum_{k=0}^{T-t} \gamma^k r_{t+k}$$

**Monte Carlo return:** compute $G_t$ backwards from the end of the episode — exact but high variance.

### Bellman Equations
Value function: $V^\pi(s) = \mathbb{E}_\pi[G_t | s_t = s]$  
Action-value:   $Q^\pi(s,a) = \mathbb{E}_\pi[G_t | s_t=s, a_t=a]$  

Bellman consistency:
$$V^\pi(s) = \sum_a \pi(a|s)\sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V^\pi(s')]$$

**Advantage:** $A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$ — how much better is action $a$ vs the average?


In [ ]:
def compute_returns(rewards: List[float], gamma: float) -> List[float]:
    """
    Discounted returns G_t = r_t + γ*r_{t+1} + γ²*r_{t+2} + ...
    Hint: single backward pass — G = 0, then G = r_t + gamma * G
    """
    # TODO
    raise NotImplementedError


def normalize_returns(returns: List[float], eps: float = 1e-8) -> List[float]:
    """Zero-mean, unit-variance normalisation of returns."""
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
rewards = [1.0, 2.0, 3.0]
assert compute_returns(rewards, gamma=0.0) == [1.0, 2.0, 3.0]
assert compute_returns(rewards, gamma=1.0) == [6.0, 5.0, 3.0]
g = compute_returns(rewards, gamma=0.9)
assert abs(g[0] - (1 + 0.9*2 + 0.81*3)) < 1e-5
normed = normalize_returns(g)
assert abs(np.mean(normed)) < 1e-5 and abs(np.std(normed) - 1.0) < 1e-5
print("compute_returns   ✓")
print("normalize_returns ✓")


---
## 2 · Policy Gradient Theorem & REINFORCE

### Policy Gradient Theorem (Williams, 1992)
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_t G_t \nabla_\theta \log \pi_\theta(a_t|s_t)\right]$$

We minimise the **negative** expected return:
$$\mathcal{L}_{PG} = -\sum_t G_t \log \pi_\theta(a_t|s_t)$$

**Variance reduction:** replacing $G_t$ with the **advantage** $A_t = G_t - b(s_t)$ where $b$ is a baseline (e.g., $V(s_t)$) leaves the gradient unbiased but reduces variance dramatically.

### REINFORCE with baseline algorithm
```
for each episode:
    collect trajectory τ = {s₀,a₀,r₀, ..., s_T,a_T,r_T}
    compute G_t for all t  (Monte Carlo)
    compute A_t = G_t - V(s_t)          # baseline subtraction
    loss = -∑_t A_t · log π(a_t|s_t)
    loss.backward(); optimizer.step()
```

### Policy networks
A discrete policy outputs a **categorical distribution** over actions: `logits → softmax → sample`.  
Key: `log_prob = log_softmax(logits)[action]` — the log-prob of the *chosen* action.


In [ ]:
class DiscretePolicy(nn.Module):
    """MLP policy: obs → Categorical distribution over actions."""
    def __init__(self, obs_dim: int, act_dim: int, hidden: int = 64):
        super().__init__()
        # TODO: define self.net — 2-hidden-layer MLP with Tanh, output dim = act_dim
        raise NotImplementedError

    def forward(self, obs: torch.Tensor) -> torch.distributions.Categorical:
        # TODO: pass obs through net → Categorical(logits=...)
        raise NotImplementedError

    def act(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample action, return (action, log_prob)."""
        # TODO: use dist.sample() and dist.log_prob()
        raise NotImplementedError


def reinforce_loss(
    log_probs: torch.Tensor,   # (T,)
    advantages: torch.Tensor,  # (T,) — detached
) -> torch.Tensor:
    """L = -mean(A_t * log π(a_t|s_t))"""
    # TODO
    raise NotImplementedError


def entropy_bonus(dist: torch.distributions.Categorical, coef: float = 0.01) -> torch.Tensor:
    """
    -coef * H(π).  Subtracting from loss encourages exploration.
    Hint: dist.entropy() gives per-sample entropy.
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
obs_dim, act_dim = 4, 2
policy  = DiscretePolicy(obs_dim, act_dim)
obs     = torch.randn(8, obs_dim)
actions, lp = policy.act(obs)
assert actions.shape == (8,) and lp.shape == (8,)
assert (lp <= 0).all(), "log-probs must be ≤ 0"
adv  = torch.tensor([1.0, -0.5, 0.3, 0.8, -1.0, 0.2, 0.4, -0.3])
loss = reinforce_loss(lp, adv)
assert loss.shape == ()
print(f"DiscretePolicy ✓  reinforce_loss ✓  entropy_bonus ✓")


---
## 3 · Value Function, TD Error & Advantage

### Value network
A separate MLP $V_\phi(s) \approx V^\pi(s)$ trained via regression on Monte Carlo returns:
$$\mathcal{L}_V = \frac{1}{T}\sum_t (V_\phi(s_t) - G_t)^2$$

### TD(0) one-step error (temporal difference)
$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This is a biased but **low-variance** advantage estimate.

### Generalised Advantage Estimation (GAE, Schulman 2015)
Interpolates between Monte Carlo (low bias, high variance) and TD(0) (high bias, low variance):

$$\hat{A}_t^{GAE(\gamma,\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$$

- $\lambda=1$: reduces to Monte Carlo advantage (high variance)
- $\lambda=0$: reduces to TD(0) error (high bias)
- $\lambda \approx 0.95$: sweet spot used in PPO and most modern algorithms

**Efficient computation** (backward pass, exactly like returns):
$$\hat{A}_T = \delta_T, \quad \hat{A}_t = \delta_t + \gamma\lambda \hat{A}_{t+1}$$


In [ ]:
class ValueNetwork(nn.Module):
    """Critic: obs → scalar V(s). Same architecture as DiscretePolicy but output dim = 1."""
    def __init__(self, obs_dim: int, hidden: int = 64):
        super().__init__()
        # TODO: 2-hidden-layer MLP with Tanh, output 1 scalar
        raise NotImplementedError

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        # TODO: return (B,) — squeeze the final dim
        raise NotImplementedError


def value_loss(values: torch.Tensor, returns: torch.Tensor) -> torch.Tensor:
    """MSE between V(s_t) predictions and MC returns G_t."""
    # TODO
    raise NotImplementedError


def compute_td_errors(
    rewards:     torch.Tensor,  # (T,)
    values:      torch.Tensor,  # (T,)  detached
    next_values: torch.Tensor,  # (T,)  V(s_{t+1}); 0 at terminal
    gamma: float,
) -> torch.Tensor:
    """δ_t = r_t + γ·V(s_{t+1}) - V(s_t)"""
    # TODO
    raise NotImplementedError


def compute_gae(
    rewards:     torch.Tensor,  # (T,)
    values:      torch.Tensor,  # (T,)  detached
    next_values: torch.Tensor,  # (T,)  detached
    gamma: float,
    lam:   float,
) -> torch.Tensor:              # (T,)
    """
    GAE backward pass:
      deltas = compute_td_errors(...)
      gae = 0
      for t in reversed(range(T)):
          gae = deltas[t] + gamma * lam * gae
          adv[t] = gae
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
T, obs_dim = 10, 4
critic = ValueNetwork(obs_dim)
obs    = torch.randn(T, obs_dim)
vals   = critic(obs)
assert vals.shape == (T,)
rewards   = torch.ones(T)
next_vals = torch.zeros(T)
next_vals[:T-1] = vals[1:].detach()
td       = compute_td_errors(rewards, vals.detach(), next_vals.detach(), gamma=0.99)
gae_td0  = compute_gae(rewards, vals.detach(), next_vals.detach(), gamma=0.99, lam=0.0)
assert torch.allclose(gae_td0, td, atol=1e-5), "GAE(λ=0) must equal TD error"
print("ValueNetwork ✓  compute_td_errors ✓  compute_gae ✓")


---
## 4 · Proximal Policy Optimization (PPO)

PPO (Schulman et al., 2017) is the direct ancestor of GRPO and the dominant deep RL algorithm.

### Key insight: trust region without second-order optimisation
Instead of constraining the KL divergence explicitly (TRPO), PPO clips the probability ratio:

$$\rho_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$$

$$\mathcal{L}^{CLIP} = -\mathbb{E}_t\left[\min\left(\rho_t A_t,\ \text{clip}(\rho_t, 1-\varepsilon, 1+\varepsilon)A_t\right)\right]$$

### Why clipping works
- If $A_t > 0$ (good action): cap ratio at $1+\varepsilon$ — don't increase probability too aggressively
- If $A_t < 0$ (bad action): floor ratio at $1-\varepsilon$ — don't decrease probability too aggressively
- The `min` ensures we take the **pessimistic** bound — always conservative

### Full PPO loss
$$\mathcal{L} = \mathcal{L}^{CLIP} + c_1 \mathcal{L}^V - c_2 H(\pi_\theta)$$

where $c_1 \approx 0.5$ (value coeff), $c_2 \approx 0.01$ (entropy coeff).

### PPO training loop
```
Collect N steps of experience with π_θ_old
Compute GAE advantages and returns
For K epochs:
    For each minibatch:
        Compute ρ, clip loss, value loss, entropy
        Update θ with gradient descent
```


In [ ]:
def ppo_clip_loss(
    log_probs_new: torch.Tensor,   # (B,)
    log_probs_old: torch.Tensor,   # (B,)  detached
    advantages:    torch.Tensor,   # (B,)  detached
    epsilon: float = 0.2,
) -> torch.Tensor:
    """
    Steps:
      ratio = exp(logp_new - logp_old.detach())
      surr1 = ratio * advantages
      surr2 = clamp(ratio, 1-ε, 1+ε) * advantages
      return -min(surr1, surr2).mean()
    """
    # TODO
    raise NotImplementedError


def ppo_loss(
    log_probs_new: torch.Tensor,
    log_probs_old: torch.Tensor,
    values:        torch.Tensor,   # (B,)  with grad
    returns:       torch.Tensor,   # (B,)  detached
    advantages:    torch.Tensor,   # (B,)  detached
    entropy:       torch.Tensor,   # scalar
    epsilon:  float = 0.2,
    vf_coef:  float = 0.5,
    ent_coef: float = 0.01,
) -> Tuple[torch.Tensor, dict]:
    """
    L = L_clip + vf_coef * L_value - ent_coef * entropy
    Return (total_loss, dict of components)
    """
    # TODO
    raise NotImplementedError


def compute_clip_fraction(
    log_probs_new: torch.Tensor,
    log_probs_old: torch.Tensor,
    epsilon: float = 0.2,
) -> float:
    """
    Fraction of samples where |ratio - 1| > epsilon.
    Use .detach() — this is a monitoring metric, not a loss.
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B      = 16
lp_new = torch.randn(B, requires_grad=True)
lp_old = lp_new.detach().clone()
adv    = torch.randn(B)
vals   = torch.randn(B, requires_grad=True)
returns= torch.randn(B)

loss_same = ppo_clip_loss(lp_new, lp_old, adv)
assert abs(loss_same.item() - (-adv.mean().item())) < 1e-5, "ratio=1 → loss == -mean(A)"

cf = compute_clip_fraction(lp_new, lp_old)
assert cf == 0.0

lp_drifted = lp_old + 1.0
cf_high = compute_clip_fraction(lp_drifted, lp_old)
assert cf_high > 0.5

dist  = torch.distributions.Categorical(logits=torch.randn(B, 4))
total, info = ppo_loss(lp_new, lp_old, vals, returns, adv, dist.entropy().mean())
total.backward()
assert lp_new.grad is not None and vals.grad is not None

print(f"ppo_clip_loss         ✓")
print(f"compute_clip_fraction ✓  same={cf:.2f}  drifted={cf_high:.2f}")
print(f"ppo_loss              ✓  {info}")


---
## 5 · A2C Training Loop on a Toy MDP

We put everything together in an **Actor-Critic (A2C)** loop on a simple GridWorld MDP.

### A2C vs REINFORCE
| | REINFORCE | A2C |
|---|---|---|
| Advantage | MC returns | TD error or GAE |
| Variance | High | Lower |
| Bias | None (MC) | Some (bootstrapping) |
| Update | Per episode | Per step or batch |

### Toy MDP: Linear Chain
- States: 0, 1, ..., N-1
- Actions: 0 (left), 1 (right)
- Reward: +1 if reaching state N-1, else 0
- Episode ends at states 0 or N-1
- Optimal policy: always go right

We test that training reduces loss and improves the policy's preference for right actions.


In [ ]:
class LinearChainMDP:
    """Chain MDP — do not modify."""
    def __init__(self, N: int = 6):
        self.N = N; self.state = N // 2
    def reset(self) -> int:
        self.state = self.N // 2; return self.state
    def step(self, action: int) -> Tuple[int, float, bool]:
        self.state += (1 if action == 1 else -1)
        self.state  = max(0, min(self.N - 1, self.state))
        done        = self.state in (0, self.N - 1)
        return self.state, (1.0 if self.state == self.N - 1 else 0.0), done


def collect_episode(env: LinearChainMDP, policy: DiscretePolicy) -> dict:
    """Roll out one episode. Return dict with obs, actions, log_probs, rewards as tensors."""
    # TODO: reset env, loop until done, collect (obs, action, log_prob, reward) per step
    raise NotImplementedError


def a2c_step(
    policy:    DiscretePolicy,
    critic:    ValueNetwork,
    optimizer: torch.optim.Optimizer,
    episode:   dict,
    gamma: float = 0.99,
    lam:   float = 0.95,
    vf_coef:  float = 0.5,
    ent_coef: float = 0.01,
) -> dict:
    """
    One A2C gradient step.
    Steps:
      1. critic(obs) → values; build next_values (shift by 1, terminal=0)
      2. compute_returns for value targets
      3. compute_gae for advantages; normalise
      4. recompute log_probs and entropy with CURRENT policy
      5. pg_loss   = reinforce_loss(log_probs_new, adv)
         vf_loss   = vf_coef * value_loss(values, returns)
         ent_loss  = ent_coef * dist.entropy().mean()
         total     = pg_loss + vf_loss - ent_loss
      6. zero_grad → backward → clip_grad_norm_(0.5) → step
    Return dict of metrics.
    """
    # TODO
    raise NotImplementedError


# ── TRAINING RUN (after your implementations pass) ─────────────────────────
torch.manual_seed(42)
env    = LinearChainMDP(N=6)
policy = DiscretePolicy(obs_dim=1, act_dim=2, hidden=32)
critic = ValueNetwork(obs_dim=1, hidden=32)
optim  = torch.optim.Adam(list(policy.parameters()) + list(critic.parameters()), lr=3e-3)

reward_history = []
for ep in range(1, 301):
    episode = collect_episode(env, policy)
    metrics = a2c_step(policy, critic, optim, episode)
    reward_history.append(metrics["episode_reward"])
    if ep % 100 == 0:
        print(f"Ep {ep:4d}  avg_reward={np.mean(reward_history[-100:]):.3f}")

obs_t  = torch.tensor([3.0])
right_prob = policy(obs_t).probs[1].item()
print(f"\nP(right | state=3) = {right_prob:.3f}  (should be > 0.6)")
assert right_prob > 0.6, f"Policy failed, P(right)={right_prob:.3f}"
print("A2C training ✓")
